## Inspect TLC Taxi Zone Lookup

In [0]:
from pyspark.sql import functions as F

zone_path = (
    "abfss://reference@stnyctaxide2026nandi.dfs.core.windows.net/"
    "taxi_zone/taxi_zone_lookup.csv"
)

df_zone = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(zone_path)
)

display(df_zone)

Inspect schema and quality

In [0]:
df_zone.printSchema()

print(f"Rows: {df_zone.count():,}")

print("Null counts:")
display(
    df_zone.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_zone.columns
    ])
)

## Check LocationID uniqueness

In [0]:
print(f"Distinct LocationID: {df_zone.select('LocationID').distinct().count():,}")

display(
    df_zone.groupBy("LocationID")
           .count()
           .filter(F.col("count") > 1)
)

## Standardize the lookup

In [0]:
df_zone_clean = (
    df_zone
    .select(
        F.col("LocationID").cast("int").alias("location_id"),
        F.trim(F.col("Borough")).alias("borough"),
        F.trim(F.col("Zone")).alias("zone"),
        F.trim(F.col("service_zone")).alias("service_zone")
    )
)

display(df_zone_clean)

## Create change-detection hash

In [0]:
df_zone_scd = (
    df_zone_clean
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("borough"), F.lit("")),
                F.coalesce(F.col("zone"), F.lit("")),
                F.coalesce(F.col("service_zone"), F.lit(""))
            ),
            256
        )
    )
)

display(df_zone_scd)

## Add initial SCD2 metadata

In [0]:
from pyspark.sql import functions as F

df_zone_initial = (
    df_zone_scd
    .withColumn("zone_sk", F.monotonically_increasing_id())
    .withColumn("effective_from", F.current_date())
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

display(df_zone_initial)

## Create Delta Table

In [0]:
gold_zone_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "dimensions/taxi_zone"
)

(
    df_zone_initial
    .select(
        "zone_sk",
        "location_id",
        "borough",
        "zone",
        "service_zone",
        "effective_from",
        "effective_to",
        "is_current",
        "record_hash",
        "created_at",
        "updated_at"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_zone_path)
)